# 🏗️ LLM 2권 · 구조 (Attention · 옴니 · 파인튜닝) 종합 정리 노트

> **생성형 AI 기반 음성 에이전트 개발 과정 · LLM 챕터 2/3 — 모델의 내부 구조 리뷰**
> `LLM/` 폴더 실습 노트북 24개를 **3권 체계**로 정리한 복습·재사용용 노트의 **2권**입니다.

| 항목 | 내용 |
|---|---|
| 이 권의 대상 | **구조 4종** — 5-T 어텐션(밑바닥) · 7-O 옴니 단일모델 · 5-V 음성 직접 입력 · 8-T 파인튜닝 |
| 노트의 목적 | ① **선행 지식** ② **함수/클래스** 정의·주석 ③ **실험 진행 방법** ④ **효율적 설계 아키텍처** |
| 실행 환경 | **macOS (Apple Silicon M4 Pro 48GB)** — 5-T는 CPU/MPS 완전 실행 |
| 나머지 권 | **1권=사고부**(Thinking) · **3권=도구**(MCP·함수호출·async·decorator) |

> ⚙️ **실행 안내** — 이 노트의 코드 셀은 **GPU·모델 다운로드·API 키 없이** 실행됩니다.
> 5-T(어텐션)는 순수 `numpy`/`torch` 밑바닥이라 **어디서든 실제 실행** ✅. 7-O/5-V/8-T는 계약·가드·계측 로직을
> 시뮬레이터로 재현하고, 실물 모델(Qwen2.5-Omni, Gemma4, MiDM) 호출부는 시그니처+가이드로 요약했습니다 (📄).


## 📑 목차
| 장 | 내용 |
|---|---|
| **0** | 노트북 로드맵 — 구조 4종 지도 · macOS 실행 판정 |
| **1** | 실험에 필요한 선행 지식 (어텐션·옴니·음성입력·파인튜닝) |
| **2** | 함수/클래스 정의 및 주석 (실행 코드 ✅ + 요약 📄) |
| **3** | 실험 진행 방법 (밑바닥 구현·모델 실습 + macOS 가이드) |
| **4** | 효율적 설계를 위한 아키텍처 |


# 0. 노트북 로드맵 🗺️

## 0-1. 구조 4종 한눈에

| 노트북 | 주제 | **M4 Pro 판정** | 핵심 포인트 |
|---|---|---|---|
| **5-T** | Transformer 밑바닥 구현 | ✅ CPU/MPS 완전 실행 | 어텐션·√dk·인과마스크·MHA·PE·base 65M 검산 |
| **7-O** | 옴니 단일 모델 | ✅ 계약 로직 / ⚠️ 모델은 GPU | text/audio 2단계, 음성 폐기 비용 |
| **5-V** | Gemma4 음성 직접 입력 | ✅ 계약 로직 / ⚠️ 모델은 GPU | 캐스케이드 없는 단일 호출, 배치 가드 |
| **8-T** | MiDM 파인튜닝 | ⚠️ QLoRA(bnb) 불가 → fp16 LoRA 대안 | 라벨 마스킹 V2, 계약 통과율 이동 |

> **구조 3부작 흐름**: 5-T(내부 원리) → 5-V/7-O(멀티모달 입력·출력) → 8-T(가중치에 스타일 새기기).


## 📖 0-A. 용어 사전 & 배경 지식 — 이 노트를 처음 읽는 사람을 위한 지도

> **이 노트를 처음 공부하는 방법**: ① 0-A 용어사전 훑기 → ② 1장 선행 지식 → ③ 2장 함수 실행하며
> "검증 통과 ✅" 눈으로 확인. 모르는 단어는 여기로 돌아오세요.

### A. 어텐션 — "모델이 무엇을 보는가"
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| 어텐션 (Attention) | Q·K 내적으로 "어디를 봐야 하는지" 가중치 계산 | 트랜스포머의 심장 |
| Q / K / V | 질의 / 키 / 값 — 각 토큰의 역할 벡터 | 내적으로 "관련도"를 재는 재료 |
| √dk 스케일 | 내적 분산이 d_k라 softmax 포화 → √로 나눔 | **수식이 아니라 통계** (노트 2.0에서 재현) |
| 인과 마스크 | 미래를 못 보게 i>j 위치를 -inf로 | 언어 모델의 "미래 차단" |
| softmax | 벡터를 확률분포로 (합=1) | 가중치 합의 규칙 |

### B. MHA · 위치 · 파라미터
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| Multi-Head | d_model을 h등분해 병렬 어텐션 | 다양한 관계를 동시에 학습 |
| 헤드 분할 | 임베딩 차원을 연속 블록으로 h등분 | torch와 같은 규약 |
| 위치 인코딩 (PE) | sin/cos으로 토큰 순서 정보 부여 | 순서 없는 임베딩에 '위치' 심기 |
| base 65M | 논문 base 모델 파라미터 수 | 노트에서 공식으로 재검산 |
| Pre-LN | 블록 앞에서 LayerNorm | 학습 안정성 |

### C. 옴니 · 멀티모달
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| 옴니 (omni) | ASR+LLM+TTS를 한 몸에 담은 모델 | 파이프라인 계약이 필요 없음 |
| generation_mode=text | 1단계: 음성→텍스트 | 7-O의 텍스트 응답 |
| generation_mode=audio | 2단계: 음성→음성 | 7-O의 음성 응답 |
| 음성 폐기 비용 | 출력 위반 시 wav 통째로 버림 | 옴니의 숨은 원가 |
| 직접 음성 입력 | 오디오를 프롬프트에 포함 | Whisper 없이 단일 호출 (5-V) |

### D. 파인튜닝
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| QLoRA | NF4 양자화 + LoRA 어댑터 | 큰 모델을 적은 VRAM으로 튜닝 |
| 라벨 마스킹 | 프롬프트 부분을 -100(손실 제외) | 모델은 '완성'만 학습 |
| LoRA | 소수의 어댑터 가중치만 학습 | 수십 MB 어댑터로 배포 |
| 계약 통과율 | 튜닝 전/후 응답 계약 준수율 | "프롬프트·가드·가중치 중 무엇이 옳았나" |

### E. 배경 지식 — 이 챕터가 왜 존재하는가
1권(사고부)이 "LLM을 부르는 법"이라면, **2권(구조)은 "LLM 안이 어떻게 생겼는지"**를 다룹니다.
1. **어텐션을 밑바닥으로 재현** — Q@K.T/√dk를 직접 쓰면 "√dk는 왜 필요한가"가 숫자로 보인다.
2. **멀티모달의 두 길** — 캐스케이드(ASR→LLM→TTS) vs 옴니(한 모델)의 트레이드오프.
3. **튜닝은 스타일 주입** — 가중치에 응답 스타일을 새기고, 통과율로 판정한다.


### 0-B. 2장 함수 지도 — 어떤 셀이 무슨 역할인지 미리 보기
| 셀 | 함수/클래스 | 역할 한 줄 | 핵심 개념 |
|---|---|---|---|
| 2.0 | `attention_scratch` | 식(1) + √dk 재현 + 인과 마스크 | 어텐션의 수식과 이유 |
| 2.1 | `MHAScratch`·`sinusoidal_pe`·`count_transformer_params` | MHA + PE + 65M 검산 | 부품 재현 |
| 2.2 | `Block`·`MiniLM` | 미니 LM + 어텐션 맵 | 실제 forward로 맵 확인 |
| 2.3 | `cer`·`validate_llm_response`·`guarded_omni_speech` | 옴니 2단계 계약 | 음성 폐기 비용 |
| 2.4 | `placement_guard`·`sanity_check_audio` | 음성 직접 입력 가드 | meta 방치·전사 괴리 |
| 2.5 | `encode_example`·`mask_ratio`·`contract_pass_rate` | 파인튜닝 계약 | -100 마스킹·통과율 |


# 1. 실험에 필요한 선행 지식 🧠

## 1-1. Scaled Dot-Product Attention (5-T) — 논문 §3.2.1

```
Attention(Q, K, V) = softmax(QKᵀ / √d_k) V
```

- **√d_k의 이유(논문 각주 4)**: 차원 d_k가 크면 Q·K 내적의 분산이 d_k로 커져 softmax가 포화
  (기울기 소실)된다. √d_k로 나눠 분산을 1로 유지 → 본 노트에서 **수치로 재현**.
- **인과 마스크**: 언어 모델은 미래를 보면 안 된다. `triu(1)` 마스크로 i>j 위치를 -inf로.

## 1-2. Multi-Head Attention · 위치 인코딩 · base 65M (5-T)

- **MHA (§3.2.2)**: d_model을 h개 헤드로 분할(각 dh=d_model/h) → 병렬 어텐션 → concat → Wo.
- **위치 인코딩 (§3.5)**: sin/cos — 짝수 차원 sin, 홀수 차원 cos, 주기 10000^(2i/d).
- **base 65M 검산**: `count_transformer_params`로 논문 주장을 공식으로 직접 재현 (embed 3중 공유 §3.4).

## 1-3. 옴니 단일 모델 (7-O) — ASR+LLM+TTS를 한 몸에

- `generation_mode="text"` = 1단계(음성→텍스트) · `generation_mode="audio"` = 2단계(음성→음성).
- **파이프라인 vs 옴니**: 옴니는 ASR/LLM/TTS 사이 계약 레코드가 필요 없다 (한 모델). 대신 **출력 계약 위반 시
  합성된 음성 wav가 통째로 폐기**되는 비용이 생긴다 — guarded 재시도에서 `total_sec`로 누적 측정.
- **재협상 테이블**: 지연(직렬 vs 단일) · 통제권(중간 레코드 유무) · 원가.

## 1-4. 음성 직접 입력 (5-V) — 캐스케이드 없는 로컬 사고부

- Gemma4에 **오디오를 직접 프롬프트에 포함** — Whisper ASR 없이 음성→JSON 단일 호출.
- **배치 가드** `placement_guard`: 로드 후 `device.type == "meta"`로 남은 파라미터가 있으면 즉시 실패
  (meta 방치 = 배치 실패). **"설치된 휠이 곧 사양"** 사전 검증 기록의 연장.
- `sanity_check_audio`: 기존 sanity_check에 **CER 괴리** 검사를 추가 — 전사가 참조와 너무 다르면 실패.

## 1-5. 파인튜닝 (8-T) — 응답 스타일을 가중치에 새긴다

- **QLoRA**: NF4 양자화 + LoRA 어댑터 — 4.6GB 베이스 대신 수십 MB 어댑터만 배포. **bnb=CUDA 전용 → macOS는
  fp16 + LoRA(peft) 대안**.
- **라벨 마스킹 V2**: `labels`에서 프롬프트 부분을 `-100`으로 — 모델은 "완성(completion)만" 학습하고
  "멈추는 법(eos)"도 학습 대상 (`encode_example`이 V2를 증명하는 줄).
- **계약 통과율 이동**: 튜닝 전/후 `contract_pass_rate` — 프롬프트·가드·가중치 중 무엇이 옳았는지 판정.
- **일반화 점검**: 학습 데이터에 없는 발화도 통과율이 유지되는지 (학습인가 암기인가).

## 1-6. macOS(M4 Pro 48GB) 실행 가이드

| 항목 | 판정 | 설명 |
|---|---|---|
| 5-T 어텐션 | ✅ 전부 실행 | numpy·torch CPU — 이 노트 2.0~2.2 셀 자체가 실습 |
| 7-O Qwen2.5-Omni-3B | ⚠️ GPU 권장 | fp16 ~6GB지만 **생성 방식**(text/audio)이 MPS에서 불안정할 수 있음 → Colab T4 권장 |
| 5-V Gemma4-E2B | ⚠️ GPU 권장 | bnb 제거·fp16 로드는 가능하나 오디오 토큰 처리량 고려 |
| 8-T MiDM 파인튜닝 | ⚠️ bnb 불가 | QLoRA는 CUDA 전용 → **fp16 LoRA**(peft) 또는 Colab |

> **핵심**: 이 권의 **계약·가드·계측 로직은 전부 macOS에서 실행 가능** — 2.0~2.5 셀에 포함.
> 실물 모델 학습/생성만 Colab(T4) 전제로 남는다.


# 2. 함수/클래스 정의 및 주석 🔧

> ✅ = 실행 코드 셀 (GPU·모델·키 없이 assert 자가점검) · 📄 = 요약만 (실물 호출은 3장 가이드)


In [ ]:
# ═══ 2.0 어텐션 스크래치 — 식(1) · √dk 재현 · 인과 마스크 (5-T) ✅ ═══
# ▶ attention_scratch: Q@K.T/√dk → (선택) 마스크 → 안정 softmax → V 가중합.
#   √dk 재현: 내적 분산이 d_k임을 2000회 샘플로 확인 — '왜 나누는가'가 숫자로 증명된다.
#   인과 마스크: 하삼각만 열면 '미래 차단'이 실제로 0이 되는지 assert로 확인.
import numpy as np

def attention_scratch(Q, K, V, mask=None):
    # 논문 식 (1) 그대로. Q,K,V: (..., T, d) numpy
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-2, -1) / np.sqrt(d_k)          # (..., T, T)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)                # 막힌 곳은 -inf 근사
    w = np.exp(scores - scores.max(axis=-1, keepdims=True))  # 안정 softmax
    w = w / w.sum(axis=-1, keepdims=True)
    return w @ V, w

np.random.seed(0)
Q = np.random.randn(1, 4, 8); K = np.random.randn(1, 4, 8); V = np.random.randn(1, 4, 8)
out, w = attention_scratch(Q, K, V)
assert out.shape == (1, 4, 8) and abs(w.sum(-1) - 1).max() < 1e-6

# ── √dk 재현: 내적 분산은 d_k다 — 스케일 없으면 softmax 포화 (논문 각주 4) ──
d_k = 64
deltas = []
for _ in range(2000):
    q = np.random.randn(1, d_k); k = np.random.randn(1, d_k)
    deltas.append(float((q @ k.T)[0, 0]))
raw_var = float(np.var(deltas))
scaled_var = raw_var / d_k
assert abs(raw_var - d_k) < d_k * 0.2, f"원시 내적 분산 ≈ d_k: {raw_var:.1f}"
assert abs(scaled_var - 1.0) < 0.2, f"스케일 후 분산 ≈ 1: {scaled_var:.2f}"

# ── 인과 마스크: i>j 위치만 열린다 ──
causal = np.tril(np.ones((4, 4), dtype=bool))               # 하삼각 True
out, w = attention_scratch(Q, K, V, mask=causal)
assert abs(w[0, 1, 2:]).max() < 1e-6                        # 1번 행: j>=2 는 막힘
print(f"어텐션 스크래치 검증 통과 ✅ — 원시 내적 분산 {raw_var:.1f}≈d_k({d_k}) → 스케일 후 {scaled_var:.2f}≈1")


In [ ]:
# ▶ 데모 — "√dk는 왜 필요한가"를 작은 숫자로 눈으로 확인 (초보자용)
import numpy as np
np.random.seed(1)

d_k = 16
# Q와 K를 랜덤(평균 0)으로 만들면, 내적은 평균 0·분산 d_k 인 값들이 나온다.
q = np.random.randn(5, d_k); k = np.random.randn(5, d_k)
raw = (q @ k.T)                       # 스케일 전 내적
scaled = raw / np.sqrt(d_k)           # 스케일 후

print(f"원시 내적: 평균 {raw.mean():+.2f}, 분산 {raw.var():.1f} (≈ d_k={d_k})")
print(f"√dk 후   : 평균 {scaled.mean():+.2f}, 분산 {scaled.var():.2f} (≈ 1)")

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

# 스케일 없으면 softmax가 한 값으로 '포화' (0과 1로 극단) — 기울기가 죽는다
s_none = softmax(raw[0])
s_scaled = softmax(scaled[0])
print(f"포화 정도 (최대 확률): 스케일 없음 {s_none.max():.3f} vs √dk {s_scaled.max():.3f}")
assert abs(scaled.var() - 1.0) < 1.0        # 분산이 1 근처로 정규화됨
assert s_scaled.max() < s_none.max()        # 스케일 후 덜 극단적
print("데모 통과 ✅ — √dk는 'softmax가 포화하지 않게' 분산을 1로 맞추는 수치 안정 장치")


In [ ]:
# ═══ 2.1 MHA 스크래치 · 위치 인코딩 · base 65M 검산 (5-T) ✅ ═══
# ▶ MHAScratch: d_model을 h등분(연속 블록)해 병렬 어텐션 → concat → Wo.
#   count_transformer_params: 논문 base 65M을 공식으로 검산 — '주장이 아니라 공식'.
import math
import torch
import torch.nn as nn

class MHAScratch(nn.Module):
    # 밑바닥 MHA. torch와 동일한 헤드 분할 규약(임베딩 차원을 연속 블록으로 h등분)
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d, self.h, self.dh = d_model, n_heads, d_model // n_heads
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x, causal=False, return_attn=False):
        B, T, _ = x.shape
        def split(t):  # (B,T,d) -> (B,h,T,dh)
            return t.view(B, T, self.h, self.dh).transpose(1, 2)
        q, k, v = split(self.Wq(x)), split(self.Wk(x)), split(self.Wv(x))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.dh)
        if causal:
            m = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), 1)
            scores = scores.masked_fill(m, float("-inf"))
        attn = scores.softmax(-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, self.d)
        out = self.Wo(out)
        return (out, attn) if return_attn else out

def sinusoidal_pe(max_len, d):
    pos = np.arange(max_len)[:, None]
    i = np.arange(d)[None, :]
    angle = pos / np.power(10000.0, (2 * (i // 2)) / d)
    pe = np.zeros((max_len, d))
    pe[:, 0::2] = np.sin(angle[:, 0::2]); pe[:, 1::2] = np.cos(angle[:, 1::2])
    return pe

def count_transformer_params(d=512, d_ff=2048, L_enc=6, L_dec=6, vocab=37000):
    # 논문 "base 65M" 주장의 검산 — embed 3중 공유(§3.4) 포함
    mha  = 4 * d * d + 4 * d                 # Wq,Wk,Wv,Wo + bias
    ffn  = d * d_ff + d_ff + d_ff * d + d    # 2개 Linear + bias
    ln   = 2 * d                             # gamma, beta
    enc_layer = mha + ffn + 2 * ln           # self-attn + FFN (+LN×2)
    dec_layer = 2 * mha + ffn + 3 * ln       # self + cross + FFN (+LN×3)
    embed = vocab * d                        # 인코더·디코더·출력 softmax 3중 공유
    return embed + L_enc * enc_layer + L_dec * dec_layer

torch.manual_seed(0)
d_model, n_heads = 8, 4
mha = MHAScratch(d_model, n_heads)
x = torch.randn(2, 5, d_model)
out, attn = mha(x, causal=True, return_attn=True)
assert out.shape == (2, 5, d_model) and attn.shape == (2, n_heads, 5, 5)
assert attn[:, :, 0, 1:].max() < 1e-6                     # 인과: 첫 행은 j>0 막힘

pe = sinusoidal_pe(10, 4)
assert pe.shape == (10, 4)

p_base = count_transformer_params()                       # 512/2048/6/6/37000
p_small = count_transformer_params(d=64, d_ff=256, L_enc=1, L_dec=1, vocab=500)
assert 55e6 < p_base < 75e6, f"base 65M 검산 실패: {p_base/1e6:.1f}M"
print(f"base 파라미터 검산 통과 ✅ — {p_base/1e6:.1f}M (논문 65M 근사) / 미니 실측 {p_small/1e3:.0f}k")


In [ ]:
# ═══ 2.2 미니 Transformer LM — Pre-LN 블록 · 어텐션 맵 (5-T) ✅ ═══
# ▶ Block: Pre-LN(MHAScratch + FFN). MiniLM forward 한 번으로 인과 어텐션 맵을 본다.
#   '모델이 무엇을 보는가'는 학습 후 어텐션 맵으로 읽는다 (논문 §9).
class Block(nn.Module):
    # Pre-LN Transformer 블록 = MHAScratch(§5) + FFN. 논문 부품 그대로
    def __init__(self, d, h, d_ff):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = MHAScratch(d, h)
        self.ffn = nn.Sequential(nn.Linear(d, d_ff), nn.ReLU(), nn.Linear(d_ff, d))
        self.last_attn = None
    def forward(self, x):
        a, w = self.attn(self.ln1(x), causal=True, return_attn=True)
        self.last_attn = w.detach()
        x = x + a
        return x + self.ffn(self.ln2(x))

class MiniLM(nn.Module):
    # 밑바닥 미니 LM — 학습 없이 forward만으로 인과 어텐션 맵을 확인한다
    def __init__(self, V, d=32, h=4, L=2, d_ff=128, max_len=16):
        super().__init__()
        self.tok = nn.Embedding(V, d)
        self.register_buffer("pe", torch.tensor(sinusoidal_pe(max_len, d), dtype=torch.float32))
        self.blocks = nn.ModuleList(Block(d, h, d_ff) for _ in range(L))
        self.ln_f, self.head = nn.LayerNorm(d), nn.Linear(d, V)
    def forward(self, ids):
        x = self.tok(ids) + self.pe[: ids.size(1)]
        for b in self.blocks:
            x = b(x)
        return self.head(self.ln_f(x))

torch.manual_seed(0)
V, d, h, L = 20, 32, 4, 2
mini = MiniLM(V, d=d, h=h, L=L, d_ff=128, max_len=16)
ids = torch.randint(0, V, (1, 8))
logits = mini(ids)
assert logits.shape == (1, 8, V)
attn = mini.blocks[0].last_attn                     # (1, h, 8, 8)
assert attn.shape == (1, h, 8, 8)
assert attn[0, :, 2, 3:].max() < 1e-6               # 인과: 2번 행은 j>2 막힘
row0 = attn[0, 0, 0, :].detach().cpu().numpy()
assert abs(row0.sum() - 1) < 1e-6                   # 행 합 = 1 (softmax)
print(f"미니 LM 검증 통과 ✅ — 어텐션 맵 {tuple(attn.shape)} / 행 합 1 / 인과 마스크 동작")
print("  [참고] 콜센터 발화를 학습시키면 모델이 한국어의 무엇을 보는지 어텐션 맵으로 읽는다 (논문 §9)")


In [ ]:
# ═══ 2.3 옴니 2단계 — text/audio · 응답 계약 · 음성 폐기 비용 (7-O) ✅ ═══
# ▶ 옴니 2단계: text(1단계)와 audio(2단계) 생성 — 파이프라인 계약이 필요 없다.
#   guarded_omni_speech: 응답 계약 위반 시 '합성된 음성 wav가 통째로 폐기'되는 비용을 total_sec로 누적.
OUT_SR = 24000   # Qwen2.5-Omni 출력 샘플레이트 (모델 카드 기준)

def cer(ref, hyp):
    # 문자 오류율(CER) — 공백 제거 후 Levenshtein / len(ref). 과정 표준 지표
    r, h = ref.replace(" ", ""), hyp.replace(" ", "")
    d = np.zeros((len(r) + 1, len(h) + 1), dtype=np.int32)
    d[:, 0] = np.arange(len(r) + 1); d[0, :] = np.arange(len(h) + 1)
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            d[i, j] = min(d[i-1, j] + 1, d[i, j-1] + 1, d[i-1, j-1] + (r[i-1] != h[j-1]))
    return float(d[len(r), len(h)]) / max(1, len(r))

class ResponseContractError(RuntimeError):
    pass

RESPONSE_CONTRACT = {"max_chars": 200, "forbidden": ["주민등록번호", "카드번호"]}

def validate_llm_response(text):
    if len(text) > RESPONSE_CONTRACT["max_chars"]:
        raise ResponseContractError(f"길이 위반: {len(text)}자")
    for bad in RESPONSE_CONTRACT["forbidden"]:
        if bad in text:
            raise ResponseContractError(f"금지 표현: {bad!r}")
    return text

class MockOmni:
    # Qwen2.5-Omni 시뮬레이터 — bad_calls=k면 처음 k회 위반(금지어) 후 정상
    def __init__(self, bad_calls=0):
        self.bad_calls = bad_calls
        self.calls = 0
    def speech(self, instruction):
        self.calls += 1
        if self.calls <= self.bad_calls:
            reply = "주민등록번호를 알려주세요"
        else:
            reply = "네, 요금 내역을 확인해 안내드리겠습니다. 잠시만 기다려 주세요."
        wav = np.zeros(int(OUT_SR * (1.0 + len(reply) * 0.05)), dtype=np.float32)
        return reply, wav

def guarded_omni_speech(engine, max_retries=1):
    # 음성 위반은 wav가 통째로 폐기된다 — 재시도 비용을 total_sec로 누적
    total_cost = 0.0
    instruction = "고객 발화를 분석해 존댓말로 응답하세요."
    for attempt in range(max_retries + 1):
        reply, wav = engine.speech(instruction)
        total_cost += len(wav) / OUT_SR
        try:
            validate_llm_response(reply)
            return reply, {"attempt": attempt, "total_sec": round(total_cost, 2)}
        except ResponseContractError as e:
            instruction += f" 반드시 {RESPONSE_CONTRACT['max_chars']}자 이내로."
    return None, {"attempt": max_retries + 1, "total_sec": round(total_cost, 2), "fallback": True}

assert abs(cer("안녕하세요", "안녕하세요") - 0.0) < 1e-9
assert abs(cer("가나다라마", "가나다라") - 1/5) < 1e-9

good = guarded_omni_speech(MockOmni(0))
assert "요금 내역" in good[0] and good[1]["attempt"] == 0

retry = guarded_omni_speech(MockOmni(1))              # 1회 위반 → 음성 폐기 + 재시도
assert "요금 내역" in retry[0] and retry[1]["attempt"] == 1 and retry[1]["total_sec"] > 0

dead = guarded_omni_speech(MockOmni(99), max_retries=1)   # 재시도 소진 → fallback
assert dead[1]["fallback"] is True
print(f"옴니 2단계 검증 통과 ✅ — 재시도 시 음성 {retry[1]['total_sec']}s 폐기 / 소진 시 안전 폴백")


In [ ]:
# ═══ 2.4 음성 직접 입력 — 배치 가드 · sanity_check_audio (5-V) ✅ ═══
# ▶ placement_guard: 로드 후 meta 파라미터 방치를 즉시 실패(배치 실패 = 재시작).
#   sanity_check_audio: 한글·반복·CER 괴리 3종 — 전사가 참조와 너무 다르면 실패.
def placement_guard(m):
    # 로드 후 meta로 남은 파라미터가 있으면 즉시 실패 (배치 실패 = 재시작)
    metas = [n for n, p in m.named_parameters() if p.device.type == "meta"]
    assert not metas, (
        f"배치 실패: {len(metas)}개 파라미터 meta 방치 (예: {metas[:3]}) — 런타임 재시작 권장")
    return len(metas)

class _Realized:
    # 정상 로드 — 전 파라미터 실체화
    def named_parameters(self):
        with torch.device("cpu"):
            p = nn.Parameter(torch.randn(2))
        p.device  # cpu
        return [("w", p)]

class _Meta:
    # 배치 실패 재현 — 파라미터 하나가 meta로 방치
    def named_parameters(self):
        with torch.device("meta"):
            p = nn.Parameter(torch.empty(2))
        return [("w_meta", p)]

def sanity_check_audio(text, ref):
    # 기존 sanity_check + CER 괴리 검사 — 전사가 참조와 너무 다르면 실패
    if not text or not text.strip():
        return False, "빈 출력"
    if not any("가" <= ch <= "힣" for ch in text):
        return False, "한글 부재"
    toks = text.split()
    if len(toks) >= 8 and len(set(toks)) <= max(2, len(toks) // 6):
        return False, "반복 붕괴 의심"
    c = cer(ref, text)
    if c > 0.5:
        return False, f"전사 괴리 (CER={c:.2f})"
    return True, f"정상 (CER={c:.2f})"

assert placement_guard(_Realized()) == 0
try:
    placement_guard(_Meta())
    raise AssertionError("배치 가드 미포착")
except AssertionError:
    pass

ref = "지난달 요금이 평소보다 많이 나온 것 같아요"
ok, why = sanity_check_audio("지난달 요금이 평소보다 많이 나온 것 같아요", ref)
assert ok
bad_rep, why2 = sanity_check_audio("안녕 안녕 안녕 안녕 안녕 안녕 안녕 안녕", ref)
assert not bad_rep
bad_cer, why3 = sanity_check_audio("주문번호가 뭐예요", ref)
assert not bad_cer and "CER" in why3
print("음성 입력 검증 통과 ✅ — 배치 가드(meta 방치 포착) / 한글·반복·CER 괴리 3종")


In [ ]:
# ═══ 2.5 파인튜닝 — 라벨 마스킹 V2 · 계약 통과율 이동 (8-T) ✅ ═══
# ▶ encode_example: 프롬프트 부분을 -100으로 마스킹 — 모델은 '완성+eos'만 학습.
#   contract_pass_rate: 튜닝 전/후 응답 계약 준수율로 '무엇이 개선됐는가'를 판정.
RESPONSE_CONTRACT_FT = {"max_chars": 120, "max_sentences": 3,
                        "forbidden": ["주민등록번호", "카드번호"]}

def validate_llm_response_ft(text):
    if len(text) > RESPONSE_CONTRACT_FT["max_chars"]:
        raise ResponseContractError(f"길이 위반: {len(text)}자")
    n_sent = sum(text.count(p) for p in ".!?") or 1
    if n_sent > RESPONSE_CONTRACT_FT["max_sentences"]:
        raise ResponseContractError(f"문장 수 위반: {n_sent}문장")
    for bad in RESPONSE_CONTRACT_FT["forbidden"]:
        if bad in text:
            raise ResponseContractError(f"금지 표현: {bad!r}")
    return text

def contract_pass_rate(replies):
    # 튜닝 전/후 계약 통과율 — "프롬프트·가드·가중치 중 무엇이 옳았는가"
    ok = 0
    for uid, r in replies.items():
        try:
            validate_llm_response_ft(r); ok += 1
        except ResponseContractError:
            pass
    return ok / len(replies)

def encode_example(q, a, vocab):
    # V2 라벨 마스킹 — 프롬프트 부분은 -100(손실 제외), 완성+eos만 학습 대상
    prompt_ids = [vocab.get(c, 0) for c in q]
    completion_ids = [vocab.get(c, 0) for c in a] + [vocab.get("<eos>", 1)]
    input_ids = prompt_ids + completion_ids
    labels = [-100] * len(prompt_ids) + completion_ids          # ← V2를 이 줄이 증명한다
    return {"input_ids": input_ids, "labels": labels,
            "n_prompt": len(prompt_ids), "n_completion": len(completion_ids)}

def mask_ratio(ds, tag):
    tot = msk = 0
    for exm in ds:
        tot += len(exm["labels"]); msk += sum(1 for l in exm["labels"] if l == -100)
    ratio = msk / tot if tot else 0.0
    print(f"[{tag}] 토큰 {tot}개 중 손실 제외(-100) {msk}개 ({ratio:.0%})")
    return ratio

vocab = {"지": 10, "난": 11, "달": 12, "요": 13, "금": 14, "이": 15, "네": 16, "확": 17, "인": 18,
         "해": 19, "드": 20, "리": 21, "겠": 22, "습": 23, "니": 24, "다": 25, " ": 2, "<eos>": 1}
ex = encode_example("지난달 요금", "네 확인해 드리겠습니다", vocab)
assert len(ex["input_ids"]) == len(ex["labels"])
assert ex["labels"][:ex["n_prompt"]] == [-100] * ex["n_prompt"]   # 프롬프트 전부 -100
assert ex["labels"][-1] == vocab["<eos>"]                          # 멈추는 법도 학습
ds = [ex, encode_example("요금", "네", vocab)]
r = mask_ratio(ds, "예시")

before = contract_pass_rate({"u1": "네 알겠습니다. 주민등록번호를 알려주세요. 감사합니다. 네. 네.",
                             "u2": "네 확인해 드리겠습니다."})
after = contract_pass_rate({"u1": "네, 요금 내역을 확인해 안내드리겠습니다.",
                            "u2": "네 확인해 드리겠습니다."})
assert ex["labels"][ex["n_prompt"]] != -100          # 완성 첫 토큰은 학습 대상
assert before < after                                # 튜닝으로 계약 통과율 상승(모의)
print(f"파인튜닝 검증 통과 ✅ — 라벨 마스킹(V2) / 계약 통과율 {before:.0%} → {after:.0%}")


## 2.6 📄 요약 — 구조 4종 핵심 카드

| | 5-T 어텐션 | 7-O 옴니 | 5-V 음성입력 | 8-T 파인튜닝 |
|---|---|---|---|---|
| 핵심 산출물 | 어텐션 수식·MHA·PE·65M | text/audio 2단계 | 오디오→JSON 단일 호출 | 라벨 마스킹·통과율 |
| 계약/가드 | 인과 마스크 | 응답 계약+음성 폐기 | 배치 가드+전사 괴리 | -100 마스킹 |
| macOS | ✅ 전부 | 계약만 ✅ | 계약만 ✅ | 계약만 ✅ (bnb 불가) |

**교훈 4줄**:
1. √dk는 "내적 분산 = d_k"에서 나온다 — 수식이 아니라 통계다.
2. 인과 마스크는 "위반을 먼저 저지르고(no mask), 잡는다"로 검증한다.
3. 옴니는 파이프라인 계약 레코드가 없어지지만, **출력 위반 시 음성이 통째로 폐기**되는 원가가 온다.
4. 파인튜닝은 가중치에 스타일을 새기고 — 판정은 프롬프트·가드·가중치의 교차 시험이다.


## 2.7 [REAL] 실물 실행 — MPS 가용성 + 미니 LM forward

> 2.2의 `MiniLM` 을 **MPS(Apple GPU)** 위에서 실제로 돌립니다. 준비: `bash setup_apple_silicon.sh ml`


In [ ]:
import sys
from pathlib import Path
_here = Path.cwd() if (Path.cwd() / "aicc_env.py").exists() else Path.cwd().parent
if str(_here) not in sys.path: sys.path.insert(0, str(_here))
import aicc_env as ae

if not ae.has("torch"):
    print("torch 미설치 → 스킵.  bash setup_apple_silicon.sh ml")
else:
    import torch
    dev = "mps" if torch.backends.mps.is_available() else "cpu"
    print("torch", torch.__version__, "| 장치:", dev, "(mps = Apple GPU)")
    torch.manual_seed(0)
    V, d, h, L = 20, 32, 4, 2
    mini = MiniLM(V, d=d, h=h, L=L, d_ff=128, max_len=16).to(dev)
    ids = torch.randint(0, V, (2, 8)).to(dev)
    logits = mini(ids)
    attn = mini.blocks[0].last_attn
    print("logits", tuple(logits.shape), "| attn 맵", tuple(attn.shape))
    assert tuple(attn.shape) == (2, h, 8, 8)
    print("미니 LM MPS/CPU forward 통과 ✅ (어텐션 맵이 실기기에서도 같은 형태)")


# 3. 실험 진행 방법 🧪

## 3-1. 5-T 어텐션 — macOS에서 전부 실행 (이 노트 2.0~2.2 셀)
```
attention_scratch(식1) → √dk 분산 재현 → 인과 마스크 → MHA 헤드 분할 → PE sin/cos
→ count_transformer_params(65M 검산) → MiniLM forward로 어텐션 맵 확인
```
Colab에서 원본 5-T를 열어 **미니 LM을 실제 학습**시키면 어텐션 맵이 "한국어의 무엇을 보는가"로 바뀐다 (논문 §9).

## 3-2. 7-O / 5-V — 실물 모델은 Colab(T4), 계약은 macOS
| 단계 | macOS | Colab/T4 |
|---|---|---|
| 계약·가드·CER·배치 가드 | 이 노트 셀 실행 ✅ | — |
| 모델 로드 | — | `Qwen2.5-Omni-3B` fp16 / `Gemma-4-E2B`(HF_TOKEN) |
| 생성 | — | `omni_generate_text/speech` · `mm_generate` |

> **5-V 로드 치환**: 원본 `load_mm("4bit")`(bnb) → `fp16` + `device_map="auto"`(MPS). HF_TOKEN 필수(gated).
> **7-O 로드**: `dtype=torch.float16`, `attn_implementation` 기본. transformers≥5.14·qwen-omni-utils 필요.

## 3-3. 8-T 파인튜닝 — macOS 대안 (bnb 불가)
1. **QLoRA 경로 (원본, Colab)**: `BitsAndBytesConfig(nf4)` + `peft` LoRA — 베이스 4.6GB 대신 어댑터 수십 MB.
2. **fp16 LoRA 경로 (macOS)**: bnb 제거, 베이스를 fp16으로 로드 후 LoRA 어댑터만 학습 — 메모리는 늘지만
   로직(마스킹·평가)은 동일.
3. **판정**: 튜닝 전/후 `contract_pass_rate` + 학습 데이터 밖 발화(golden 외)로 **일반화** 확인.

## 3-4. 판단 기준
1. **어텐션 정합성** — 인과 마스크가 i>j를 0으로 만드는가 · 행 합 = 1.
2. **파라미터 검산** — base 65M이 논문과 일치하는가 (공식 대입).
3. **음성 폐기 비용** — guarded 재시도 시 `total_sec` 누적 (재시도가 비싼 이유).
4. **배치 가드** — 로드 직후 meta 파라미터 0개인가.
5. **튜닝 판정** — 통과율 이동 + 일반화(암기 아닌 학습).


# 4. 효율적 설계를 위한 아키텍처 🏛️

## 4-1. 구조 3부작의 아키텍처 흐름

```
5-T (원리) → 5-V/7-O (멀티모달) → 8-T (가중치)
  어텐션·마스크    단일모델 vs 파이프라인    라벨 마스킹·어댑터
```

## 4-2. 파이프라인 vs 옴니 — 재협상 테이블 (7-O)
| 축 | 파이프라인 (5-T/ASR+LLM+TTS) | 옴니 단일모델 (7-O) |
|---|---|---|
| 계약 레코드 | ASR/LLM/TTS 각각 검증 | 필요 없음 (한 모델) |
| 통제권 | 중간 레코드로 교체·개입 용이 | 모델 내부 통제 |
| 위반 원가 | LLM 텍스트 폐기 (저비용) | **합성된 음성 통째로 폐기** (고비용) |
| 지연 | 직렬 합산 | 단일 생성 |

## 4-3. macOS 적용 아키텍처
- **실행 가능 층(계약·가드·계측)** 과 **GPU 전용 층(모델 생성·학습)** 을 분리 — 전자는 어디서든, 후자는 Colab/T4.
- 8-T는 bnb 대신 **fp16 LoRA**: 마스킹 로직(encode_example)은 순수 로직이라 macOS에서 검증 가능.

## 4-4. 최종 판정
- **이해**: 어텐션 수식을 밑바닥으로 재현해 "모델이 무엇을 보는가"를 손으로 확인 (5-T).
- **선택**: 멀티모달 요구가 단순하면 옴니(7-O), 통제가 필요하면 파이프라인(5-T/5-V 결합).
- **개선**: 응답 스타일은 프롬프트로는 한계 → 가중치(파인튜닝)로 옮길 시점을 계약 통과율로 판정 (8-T).
